In [ ]:
#Montar mi google drive para poder usar los archivos
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
pip install --force-reinstall plotly

In [ ]:

def antiSpace(strParam):
  result = ''
  for ch in strParam:
    if ch != ' ' and ch != ',' and ch != '/' and ch != '1':
      result = result + ch
    else:
      result = result + '_'
  return result

# 1980-2022, without emerging sources downloaded January 18 2024

Data were retrieved from InCites, a bibliometric tool for research evaluation developed by Clarivate Analytics. Annual global outputs from 1980 to 2022 across the 22 research fields defined by the Essential Science Indicators (ESI) schema were downloaded using the Analyze -> Research Areas option, selecting the Trend checkbox. Country-level outputs for each ESI area were obtained via the Analyze -> Locations menu, filtering data by area under the ESI scheme and downloading 22 separate files containing country-specific information.

In [ ]:
%cd /content/gdrive/My Drive/Ciencias/Data/WoS/InCites/ESI/ManuscriptExperiments/Locations
!ls

In [ ]:

import pandas as pd
import numpy as np
import os
#
INDICADOR = 'Web of Science Documents'
YEAR = 'Publication Year'


In [ ]:
# Read the files from the areas. IMPORTANT: Set the file names.
areas = ['Agricultural Sciences', 'Biology & Biochemistry', 'Chemistry', 'Clinical Medicine', 'Computer Science', 'Economics & Business',
         'Engineering', 'Environment/Ecology', 'Geosciences', 'Immunology', 'Materials Science', 'Mathematics', 'Microbiology',
         'Molecular Biology & Genetics', 'Multidisciplinary', 'Neuroscience & Behavior', 'Pharmacology & Toxicology', 'Physics', 'Plant & Animal Science',
         'Psychiatry/Psychology', 'Social Sciences, general', 'Space Science']

fileNames = {
              'Agricultural Sciences': 'Agricultural Sciences.xlsx',
             'Biology & Biochemistry': 'Biology & Biochemistry.xlsx',
             'Chemistry': 'Chemistry.xlsx',
             'Clinical Medicine':'Clinical Medicine.xlsx'	,
             'Computer Science':'Computer Science.xlsx',
             'Economics & Business':'Economics & Business.xlsx',
             'Engineering':'Engineering.xlsx',
             'Environment/Ecology':'Environment_Ecology.xlsx',
             'Geosciences':'Geosciences.xlsx',
             'Immunology':'Immunology.xlsx',
             'Materials Science':'Materials Science.xlsx',
             'Mathematics':'Mathematics.xlsx',
             'Microbiology':'Microbiology.xlsx',
             'Molecular Biology & Genetics':'Molecular Biology & Genetics.xlsx',
             'Multidisciplinary':'Multidisciplinary.xlsx',
             'Neuroscience & Behavior':'Neuroscience & Behavior.xlsx',
             'Pharmacology & Toxicology':'Pharmacology & Toxicology.xlsx',
             'Physics':'Physics.xlsx',
             'Plant & Animal Science':'Plant & Animal Science.xlsx',
             'Psychiatry/Psychology':'Psychiatry_Psychology.xlsx',
             'Social Sciences, general':'Social Sciences, general.xlsx',
             'Space Science':'Space Science.xlsx'
            }


df_Areas = {}
for area in areas:
  fName = fileNames[area]
  if fName != None:
    df_Areas[area] = pd.read_excel(fName, index_col=0)


In [ ]:
df_Areas['Agricultural Sciences']

In [ ]:
# Generate the list of units. It is likely that all units exist in all files.
units = np.unique(df_Areas['Agricultural Sciences'].index)

for area in df_Areas:
  units_temp = np.unique(df_Areas[area].index)
  units = np.concatenate((units, units_temp), axis=None)
units = np.unique(units)
units

In [ ]:
# Create a production matrix Years × Areas for each country
df_units = {}
for unit in units:
  df_units[unit] =  pd.DataFrame()
  for area in fileNames:
    if fileNames[area] != None:
      #Dataframe con los datos de la unidad en el area
      if unit in df_Areas[area].index:
        dfUnit_Area = df_Areas[area].loc[unit]

        dfAreaUnit = dfUnit_Area[[YEAR, INDICADOR]].values
        if dfAreaUnit.size > 2:
          dfAreaUnit = pd.DataFrame({YEAR:dfAreaUnit[:,0], area:dfAreaUnit[:,1]})
          dfAreaUnit = dfAreaUnit.set_index(YEAR)
          if df_units[unit].empty:
            df_units[unit] =  dfAreaUnit
          else:
            df_units[unit] = pd.merge(df_units[unit], dfAreaUnit, on=YEAR, how='outer')
  df_units[unit].fillna(0, inplace=True)
  df_units[unit].sort_index(inplace=True)

In [ ]:
## Areas' production over the years
df_areas=pd.DataFrame(index=df_units[unit].index)

for area in areas: #Number of papers is at 'Baseline for All Items'
  df_areas[area] = df_units['Baseline for All Items'][area]

df_areas.head()

In [ ]:
años = df_areas.index

In [ ]:
# Explore a selected country's production
import matplotlib.pyplot as plt
from __future__ import print_function
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets

selected_unit = 'USA'

def plotShare(Unit):
  selected_unit=Unit
  dft=df_units[Unit]
  dft.plot(figsize=(20,10), title = Unit)
  return Unit

w = interactive(plotShare, Unit=units)
display(w)

## The following chart uses the selected country in the last cell


In [ ]:
import plotly.graph_objects as go

Unit = w.result
dft =df_units[Unit]
print(Unit)

figDocsMexico = go.Figure()
figDocsMexico.update_layout(width=1400, height=800, title=Unit+' , DOCS')
for column in dft.columns:
  figDocsMexico.add_trace( go.Scatter(x=dft.index, y=dft[column], name= column))

figDocsMexico.show()



# Indicators Calculation

In [ ]:
years = df_areas.index
years

In [ ]:
# Read the global indicators of the areas.
unitsTrend = pd.read_excel("Incites Locations Trend.xlsx", index_col=0)
unitsTrend.head()


In [ ]:
## Change directory
%cd '/content/gdrive/My Drive/Ciencias/Data/WoS/InCites/ESI/ManuscriptExperiments/Global Context/'
!ls

In [ ]:
# Annual production of the units (countries).
df_units_production = {}
df_units_production2024 = pd.DataFrame(columns = ['Documents'])

#First calculate the total production of each unit in each year
for unit in units:
  df_units_production[unit] = pd.DataFrame(index=df_units[unit].index, columns=['Documents'])
  dfT = unitsTrend.loc[[unit]]
  dfT.set_index('Publication Year', inplace=True)
  for year in df_units_production[unit].index:
    if year in dfT.index:
      df_units_production[unit].at[year, 'Documents'] = dfT.loc[year]['Web of Science Documents']
      if year == 2024:
        df_units_production2024.at[unit, 'Documents'] = dfT.loc[year]['Web of Science Documents']
    else:
      print('ERROR')

In [ ]:
df_units_production2024.at[unit, 'Documents']

In [ ]:
df_units_production2024['Countries'] = df_units_production2024.index
df_units_production2024.sort_values('Documents', ascending=False, inplace=True)
df_units_production2024

In [ ]:
import plotly.express as px


df = df_units_production2024.head(50)
df = df.drop('Global Baseline').drop('Baseline for All Items').sort_values('Documents')
fig = px.bar(df, x='Documents', y='Countries', orientation='h')

fig.update_layout(
    yaxis=dict(tickfont=dict(size=10)),
    width=800,  # Width in pixels
    height=800  # Height in pixels
)
fig.show()

In [ ]:
df = df_areas.loc[2024].sort_values(ascending=False)
df = pd.DataFrame(df)
df.rename(columns={2024:'Documents, 2024'}, inplace=True)
df['Areas'] = df.index

fig = px.bar(df, x='Documents, 2024', y='Areas', orientation='h')

fig.update_layout(
    yaxis=dict(tickfont=dict(size=12)),
    width=800,  # Width in pixels
    height=600  # Height in pixels
)
fig.show()

In [ ]:
df

In [ ]:
# Cálculo de matriz producción en el dominio sobre la producción del pais
df_units_production2024.sort_values('Documents', ascending=False, inplace=True)
df_OD_OC=pd.DataFrame(index=df_units_production2024.index, columns=df_areas.columns)
for unit in df_units_production2024.index:
  for area in df_areas.columns:
    if df_units_production2024.at[unit, 'Documents'] != 0:
      df_OD_OC.at[unit, area] = df_areas.at[2024, area] / df_units_production2024.at[unit, 'Documents']
    else:
      df_OD_OC.at[unit, area] = 0
df_OD_OC = df_OD_OC.apply(pd.to_numeric, errors='coerce') # Convert to numeric, coercing errors to NaN
df_OD_OC.dropna(axis=0, how='all',inplace=True)
df_OD_OC

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


df =df_OD_OC


# Create the heatmap
plt.figure(figsize=(36, 32)) # Adjust figure size as needed
sns.heatmap(df, annot=True, cmap='viridis', fmt=".0f", linewidths=.1)
plt.title('Heatmap from DataFrame')
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LogNorm

df =df_OD_OC


# Create the heatmap
plt.figure(figsize=(150, 100)) # Adjust figure size as needed
sns.heatmap(df, annot=True, norm=LogNorm(vmin=df.min().min(), vmax=df.max().max()), cmap="viridis")
plt.title('Heatmap from DataFrame')
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LogNorm

df =df_OD_OC


# Create the heatmap
plt.figure(figsize=(70, 50)) # Adjust figure size as needed
sns.heatmap(df, annot=True, norm=LogNorm(vmin=df.min().min(), vmax=df.max().max()), cmap="viridis")
plt.title('Heatmap from DataFrame')
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LogNorm

df =df_OD_OC


# Create the heatmap
plt.figure(figsize=(59, 35)) # Adjust figure size as needed
sns.heatmap(df, annot=True, norm=LogNorm(vmin=df.min().min(), vmax=df.max().max()), cmap="viridis")
plt.title('Heatmap from DataFrame')
plt.show()

In [ ]:
import plotly.express as px

# --- CORRECCIÓN AQUÍ ---
# Se usa 'range_color' en lugar de 'zmin' y 'zmax'
fig = px.imshow(
    df,
    text_auto=True,
    aspect="auto",
    color_continuous_scale='Viridis',
    # Define el rango de valores para la escala de colores
    range_color=[df[df > 0].min().min(), df.max().max()]
)

# 2. Actualización de la escala de color a tipo 'log'
# Esta parte sigue siendo la clave para la escala logarítmica
#fig.update_xaxes(type="log")

# 3. (Opcional) Mejorar las etiquetas de la barra de colores para que sean más legibles
fig.update_layout(
    coloraxis_colorbar={
        'title': 'Valor (Escala Log)',
        'tickvals': [0.1, 1, 10, 100, 1000, 5000], # Marcas en potencias de 10
        'ticktext': ['0.1', '1', '10', '100', '1k', '5k'], # Etiquetas para esas marcas
    }
)

fig.show()

In [ ]:
import math

def sqrtF(number):
  return math.sqrt(number)

In [ ]:

#We use Baselines for all items as the WORLD PRODUCTION
#The following lines calculate world production
df_produc = pd.DataFrame(index=años, columns=['Documents'])
df_produc['Documents'] = df_units_production['Baseline for All Items']


# Calculate the annual share of each OD/OW area
df_areas_share = df_areas.copy()
for area in areas:
  df_areas_share[area] = df_areas[area] / df_produc.Documents # OD / OW


#Calculate the share in each area of ​​each country  OCD/OC
df_units_share={} # DShareC
df_units_wshare={} # CShareD
df_geometric_aggregation={}
for unit in units:
  df_units_share[unit] = df_units[unit].copy()
  df_units_wshare[unit] = df_units[unit].copy()
  df_geometric_aggregation[unit] = df_units[unit].copy()
  for area in fileNames:
      if fileNames[area] != None:
        if area in df_units_share[unit].columns:
          df_units_share[unit][area] = df_units[unit][area] / df_units_production[unit].Documents #OCD/OC
          df_units_production[unit]['Share'] = df_units_production[unit].Documents / df_produc.Documents # Country Share
          df_units_wshare[unit][area] = df_units[unit][area] / df_areas[area] # OCD/OWD
          # Geometric aggregation, same weight (1/2)
          df_geometric_aggregation[unit][area] = np.sqrt(df_units_share[unit][area].astype(float)) + np.sqrt(df_units_wshare[unit][area].astype(float))

#Calculate the activity index (AI)
df_units_AI={}
for unit in units:
  df_units_AI[unit] = df_units_share[unit].copy()
  for area in fileNames:
      if fileNames[area] != None:
        if area in df_units_AI[unit].columns:
          df_units_AI[unit][area] = df_units_share[unit][area] / df_areas_share[area]


#Calculate F-Measure
df_units_FMeasure={}
df_units_AMeasure={}
df_units_GMeasure={}
for unit in units:
  df_units_FMeasure[unit] = df_units_share[unit].copy()
  df_units_AMeasure[unit] = df_units_share[unit].copy()
  df_units_GMeasure[unit] = df_units_share[unit].copy()
  for area in fileNames:
      if fileNames[area] != None:
        if area in df_units_FMeasure[unit].columns:
          ODmasOC = df_areas[area] + df_units_production[unit].Documents # OD + OC
          df_units_FMeasure[unit][area] = 2*df_units[unit][area] / ODmasOC # 2*Ocd/(OD+OC)
          df_units_AMeasure[unit][area] = (df_units_share[unit][area] + df_units_wshare[unit][area])/2
          df_units_GMeasure[unit][area] = list(map(sqrtF, df_units_share[unit][area] * df_units_wshare[unit][area]))# Calculate squre root for each element


#Calculate the Relative Specialization Index (RSI)
df_units_RSI = {}
for unit in units:
  df_units_RSI[unit] = df_units_AI[unit].copy()
  for area in fileNames:
      if fileNames[area] != None:
        if area in df_units_RSI[unit].columns:
          df_units_RSI[unit][area] = (df_units_AI[unit][area] - 1) / (df_units_AI[unit][area] + 1)


def concatena(dataframes):
  result = pd.DataFrame(columns = np.append(['Unit Name'], areas))
  for unit in units:
    df_unit_temp = pd.DataFrame(columns = np.append(['Unit Name'], areas))
    for area in areas:
      if area in dataframes[unit].columns:
        df_unit_temp[area] = dataframes[unit][area]
    df_unit_temp['Unit Name'] = [str(e) + '_' + unit for e in dataframes[unit].index]
    frames = [df_unit_temp, result]
    result = pd.concat(frames)
  result.fillna(0, inplace=True)
  return result

#Create a dataframe with the evolution of all units and save into files
#### DShareC
df_allUnits_Share = concatena(df_units_share)
df_allUnits_Share.to_csv('share.txt', index=False, sep=';')

###   CShareD
df_allUnits_wShare = concatena(df_units_wshare)
df_allUnits_wShare.to_csv('Wshare.txt', index=False, sep=';')

df_allUnits_AI = concatena(df_units_AI)
df_allUnits_AI.to_csv('ai.txt', index=False, sep=';')

df_allUnits_RSI = concatena(df_units_RSI)
df_allUnits_RSI.to_csv('rsi.txt', index=False, sep=';')

df_allUnits_FMeasure = concatena(df_units_FMeasure)
df_allUnits_FMeasure.to_csv('FMEASURE.txt', index=False, sep=';')

#### geometric mean
df_allUnits_GMeasure = concatena(df_units_GMeasure)
df_allUnits_GMeasure.to_csv('GMEASURE.txt', index=False, sep=';')

#### arithmetic mean
df_allUnits_AMeasure = concatena(df_units_AMeasure)
df_allUnits_AMeasure.to_csv('AMEASURE.txt', index=False, sep=';')

#### Geometric Aggregation
df_allUnits_GAggregation = concatena(df_geometric_aggregation)
df_allUnits_GAggregation.to_csv('GAggregation.txt', index=False, sep=';')

In [ ]:
import matplotlib.pyplot as plt
from __future__ import print_function
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets
selected_unit = 'USA'

def plotShare(Unit):
  selected_unit=Unit
  dft=df_units_share[Unit]
  dft.plot(figsize=(20,10), title = Unit+' - DShareC')
  return Unit

w = interactive(plotShare, Unit=units)
display(w)

In [ ]:
import plotly.graph_objects as go

Unit = w.result
dft =df_units_share[Unit]
print(Unit)

figMex = go.Figure()
figMex.update_layout(width=1400, height=800, title= Unit+' - DShareC')
for column in dft.columns:
  figMex.add_trace( go.Scatter(x=dft.index, y=dft[column], name= column))
figMex.update_layout(
yaxis = dict(
tickfont = dict(size=20)))
figMex.show()


In [ ]:
df_units_FMeasure['USA']['Clinical Medicine'].plot()

In [ ]:
temp = df_units_AI['USA']['Clinical Medicine'] - df_units_AI['USA']['Clinical Medicine'].shift()
temp.plot()

In [ ]:
serieAI = pd.Series(df_units_AI['USA']['Clinical Medicine'].values.astype('float'))
serieAI.pct_change(periods=1).dropna().plot()

In [ ]:
serieShare = pd.Series(df_units_share['USA']['Clinical Medicine'].values.astype('float'))
serieShare.pct_change(periods=1).dropna().plot()

# Correlations between indicators - Countries are the entities

In [ ]:
#Remove 'Baseline for All Items' and 'Global Baseline'
units = list(units)
units.remove('Baseline for All Items')
units.remove('Global Baseline')
#units

In [ ]:
listIndicators = ['Output','DShareC', 'CShareD', 'AI', 'RSI', 'FMeasure', 'Geometric Aggregtion']
listDfs = [df_units_production, df_units_share, df_units_wshare, df_units_AI, df_units_RSI, df_units_FMeasure,df_units_AMeasure, df_units_GMeasure, df_geometric_aggregation]

#Create dictionaries
dictCorrAreas = {}
dictCorrAreasDF = {}
dictCorrAreasDFAGMeans = {}
dictSpearmanCorrAreas = {}
dictSpearmanCorrAreasDF = {}
dictSpearmanCorrAreasDFAGMeans = {}


dictYearlyData = {}

#Create a dictionary in the dictionaries
for year in years:
  dictCorrAreas[year] = {}
  dictSpearmanCorrAreas[year] = {}
  dictCorrAreasDF[year] = pd.DataFrame(index = fileNames)
  dictCorrAreasDFAGMeans[year] = pd.DataFrame(index = fileNames)
  dictSpearmanCorrAreasDF[year] = pd.DataFrame(index = fileNames)
  dictSpearmanCorrAreasDFAGMeans[year] = pd.DataFrame(index = fileNames)
  dictYearlyData[year] = {}


  for area in fileNames:
    dfTemp = pd.DataFrame(index=units, columns = listIndicators)
    for unit in units:
      if not df_units_share[unit].empty and area in df_units_share[unit].columns and year in df_units_share[unit].index:
        dfTemp.at[unit, 'Output'] = df_units[unit][area][year].astype('float')
        dfTemp.at[unit, 'DShareC'] = df_units_share[unit][area][year].astype('float')
        dfTemp.at[unit, 'CShareD'] = df_units_wshare[unit][area][year].astype('float')
        dfTemp.at[unit, 'AI'] = df_units_AI[unit][area][year].astype('float')
        dfTemp.at[unit, 'RSI'] = df_units_RSI[unit][area][year].astype('float')
        dfTemp.at[unit, 'FMeasure'] = df_units_FMeasure[unit][area][year].astype('float')
        dfTemp.at[unit, 'AMean'] = df_units_AMeasure[unit][area][year].astype('float')
        dfTemp.at[unit, 'GMean'] = df_units_GMeasure[unit][area][year].astype('float')
        dfTemp.at[unit, 'Geometric Aggregtion'] = df_geometric_aggregation[unit][area][year].astype('float')
    dfTemp = dfTemp.dropna(how='all')
    dfTemp = dfTemp.astype('float')
    ## CORRELATION COEFFICIENTS
    dictCorrAreas[year][area] = dfTemp.corr(method='pearson')
    dictSpearmanCorrAreas[year][area] = dfTemp.corr(method='spearman')
    ## TESTS AND DATA



    #dictCorrAreasDF[year].at[area,'AI vs Output'] = dictCorrAreas[year][area].at['AI','Output']
    dictCorrAreasDF[year].at[area,'AI vs DShareC'] = dictCorrAreas[year][area].at['AI', 'DShareC']
    dictCorrAreasDF[year].at[area,'AI vs CShareD'] = dictCorrAreas[year][area].at['AI','CShareD']
    #dictCorrAreasDF[year].at[area,'F-Measure vs Output'] = dictCorrAreas[year][area].at['FMeasure','Output']
    dictCorrAreasDF[year].at[area,'F-Measure vs DShareC'] = dictCorrAreas[year][area].at['FMeasure', 'DShareC']
    dictCorrAreasDF[year].at[area,'F-Measure vs CShareD'] = dictCorrAreas[year][area].at['FMeasure', 'CShareD']
    #dictCorrAreasDF[year].at[area,'Geometric Aggregtion vs Output'] = dictCorrAreas[year][area].at['Geometric Aggregtion','Output']
    dictCorrAreasDF[year].at[area,'Geometric Aggregtion vs DShareC'] = dictCorrAreas[year][area].at['Geometric Aggregtion', 'DShareC']
    dictCorrAreasDF[year].at[area,'Geometric Aggregtion vs CShareD'] = dictCorrAreas[year][area].at['Geometric Aggregtion', 'CShareD']
    dictCorrAreasDF[year].at[area,'AI vs RSI'] = dictCorrAreas[year][area].at['AI','RSI']
    dictCorrAreasDF[year].at[area,'AI vs F-Measure'] = dictCorrAreas[year][area].at['AI','FMeasure']
    dictCorrAreasDF[year].at[area,'CShareD vs Output'] = dictCorrAreas[year][area].at['CShareD','Output']
    dictCorrAreasDF[year].at[area,'DShareC vs CShareD'] = dictCorrAreas[year][area].at['DShareC','CShareD']

    #dictSpearmanCorrAreasDF[year].at[area,'AI vs Output'] = dictSpearmanCorrAreas[year][area].at['AI','Output']
    dictSpearmanCorrAreasDF[year].at[area,'AI vs DShareC'] = dictSpearmanCorrAreas[year][area].at['AI', 'DShareC']
    dictSpearmanCorrAreasDF[year].at[area,'AI vs CShareD'] = dictSpearmanCorrAreas[year][area].at['AI','CShareD']
    #dictSpearmanCorrAreasDF[year].at[area,'F-Measure vs Output'] = dictSpearmanCorrAreas[year][area].at['FMeasure','Output']
    dictSpearmanCorrAreasDF[year].at[area,'F-Measure vs DShareC'] = dictSpearmanCorrAreas[year][area].at['FMeasure', 'DShareC']
    dictSpearmanCorrAreasDF[year].at[area,'F-Measure vs CShareD'] = dictSpearmanCorrAreas[year][area].at['FMeasure', 'CShareD']
    #dictSpearmanCorrAreasDF[year].at[area,'Geometric Aggregtion vs Output'] = dictSpearmanCorrAreas[year][area].at['Geometric Aggregtion','Output']
    dictSpearmanCorrAreasDF[year].at[area,'Geometric Aggregtion vs DShareC'] = dictSpearmanCorrAreas[year][area].at['Geometric Aggregtion', 'DShareC']
    dictSpearmanCorrAreasDF[year].at[area,'Geometric Aggregtion vs CShareD'] = dictSpearmanCorrAreas[year][area].at['Geometric Aggregtion', 'CShareD']
    dictSpearmanCorrAreasDF[year].at[area,'AI vs RSI'] = dictSpearmanCorrAreas[year][area].at['AI','RSI']
    dictSpearmanCorrAreasDF[year].at[area,'AI vs F-Measure'] = dictSpearmanCorrAreas[year][area].at['AI','FMeasure']
    dictSpearmanCorrAreasDF[year].at[area,'CShareD vs Output'] = dictSpearmanCorrAreas[year][area].at['CShareD','Output']
    dictSpearmanCorrAreasDF[year].at[area,'DShareC vs CShareD'] = dictSpearmanCorrAreas[year][area].at['DShareC','CShareD']

    #dictCorrAreasDFAGMeans[year].at[area,'A-Measure vs Output'] = dictCorrAreas[year][area].at['AMeasure','Output']
    dictCorrAreasDFAGMeans[year].at[area,'AMean vs DShareC'] = dictCorrAreas[year][area].at['AMean', 'DShareC']
    dictCorrAreasDFAGMeans[year].at[area,'AMean vs CShareD'] = dictCorrAreas[year][area].at['AMean','CShareD']
    dictCorrAreasDFAGMeans[year].at[area,'AMean vs F-Measure'] = dictCorrAreas[year][area].at['AMean','FMeasure']
    #dictCorrAreasDFAGMeans[year].at[area,'G-Measure vs Output'] = dictCorrAreas[year][area].at['GMeasure','Output']
    dictCorrAreasDFAGMeans[year].at[area,'GMean vs DShareC'] = dictCorrAreas[year][area].at['GMean', 'DShareC']
    dictCorrAreasDFAGMeans[year].at[area,'GMean vs CShareD'] = dictCorrAreas[year][area].at['GMean', 'CShareD']
    dictCorrAreasDFAGMeans[year].at[area,'GMean vs F-Measure'] = dictCorrAreas[year][area].at['GMean','FMeasure']

    #dictSpearmanCorrAreasDFAGMeans[year].at[area,'A-Measure vs Output'] = dictSpearmanCorrAreas[year][area].at['AMeasure','Output']
    dictSpearmanCorrAreasDFAGMeans[year].at[area,'AMean vs DShareC'] = dictSpearmanCorrAreas[year][area].at['AMean', 'DShareC']
    dictSpearmanCorrAreasDFAGMeans[year].at[area,'AMean vs CShareD'] = dictSpearmanCorrAreas[year][area].at['AMean','CShareD']
    dictSpearmanCorrAreasDFAGMeans[year].at[area,'AMean vs F-Measure'] = dictSpearmanCorrAreas[year][area].at['AMean','FMeasure']
    #dictSpearmanCorrAreasDFAGMeans[year].at[area,'G-Measure vs Output'] = dictSpearmanCorrAreas[year][area].at['GMeasure','Output']
    dictSpearmanCorrAreasDFAGMeans[year].at[area,'GMean vs DShareC'] = dictSpearmanCorrAreas[year][area].at['GMean', 'DShareC']
    dictSpearmanCorrAreasDFAGMeans[year].at[area,'GMean vs CShareD'] = dictSpearmanCorrAreas[year][area].at['GMean', 'CShareD']
    dictSpearmanCorrAreasDFAGMeans[year].at[area,'GMean vs F-Measure'] = dictSpearmanCorrAreas[year][area].at['GMean','FMeasure']

    dictYearlyData[year][area] = dfTemp


In [ ]:
import os
import numpy as np
from scipy.stats import shapiro
import seaborn as sns
import plotly.express as px
import matplotlib.pyplot as plt


text= '''
  <!DOCTYPE html>
  <html>
  <body>

  <br>
  <br>
  <h1>ScatterPlots</h1>
  <br>
  <br>
  <ul>
  '''


dicTests = {}
average = 0
os.makedirs('scatterplots', exist_ok = True)

for year in years:
  if year != 2024:
    continue
  os.makedirs('scatterplots/'+str(year), exist_ok = True)
  textScatterplots = '''
  <!DOCTYPE html>
  <html>
  <body>

  <br>
  <br>
  <h1>ScatterPlots '''+str(year)+'''+</h1>
  <br>
  <br>

  '''

  dicTests[year] = pd.DataFrame(index = fileNames)
  for area in fileNames:
    df = dictYearlyData[year][area].sort_values('Output', ascending=False)
    ## Select a subset indicators
    df = df[['Output','DShareC', 'CShareD', 'AI', 'FMeasure', 'RSI']]
    # Add the table to the html
    textScatterplots = textScatterplots +'''<br> <br> <h1>'''+ str(year) + " - "+area +'''</h1> <br><br>'''

    cm = sns.light_palette("lightblue", as_cmap=True)
    s = df.style.background_gradient(cmap=cm)
    textScatterplots = textScatterplots + s.to_html()
    textScatterplots = textScatterplots + '''<br><br>'''
    ## Create a pairplot with plotly
    fig = px.scatter_matrix(df,
                            dimensions=df.columns,
                            symbol=df.index,
                            title=str(year) + " - "+area)
    fig.update_traces(diagonal_visible=False)
    areaS = area.replace('/', '')
    figName = 'scatterplots/'+str(year)+'/'+str(year)+'_'+areaS
    figNamehtml = figName+'.html'
    #html
    fig.write_html(figNamehtml,
                   full_html=False,
                   include_plotlyjs='cdn')

    textScatterplots = textScatterplots + '''<a href="'''+figNamehtml.replace('scatterplots/', '')+'''">Plotly - '''+area+'''</a>'''

    textScatterplots = textScatterplots + '''
    <br><br>
    '''
    # Set a global font scale
    sns.set_theme(font_scale=1.6)
    #Create a pairplot with seaborn
    scatter_plot  = sns.pairplot(df)
    scatter_fig = scatter_plot.fig
    scatter_fig.savefig(figName+'.png')
    textScatterplots = textScatterplots + '''
    <img src="'''+figName.replace('scatterplots/', '')+'''.png" alt="'''+str(year) + ''' - '''+area+'''">
    <br><br>'''
    #Intent closing the figure
    plt.close()

    dicTests[year].at[area,'Number of Countries'] = len(df)
    for indicator in df.columns:
      stat, p = shapiro(df[indicator])
      if p > 0.05:
        dicTests[year].at[area, indicator +'-Shapiro' ] = 1
      else:
        dicTests[year].at[area, indicator +'-Shapiro' ] = 0
  ave =  dicTests[year]['Number of Countries'].mean()
  print(year, ' - ' , ave)
  average += ave
  #close html and save it
  textScatterplots = textScatterplots + '''  </body></html>'''

  file = open("scatterplots/scatterPlots_"+str(year)+".html","w")
  file.write(textScatterplots)
  file.close()
  text = text + '''<li><a href="'''+"scatterplots/scatterPlots_"+str(year)+".html"+'''">'''+str(year)+'''</a></li>'''





text = text + '''</ul> </body></html>'''
file = open("scatterplots.html","w")
file.write(text)
file.close()

average = average / len(years)
print('Average', average)


In [ ]:
import plotly.express as px

fig = px.scatter_matrix(dfTemp,
    dimensions=dfTemp.columns,
                        symbol=dfTemp.index,
    title="Scatter matrix") # remove underscore
fig.update_traces(diagonal_visible=False)


fig.show()

In [ ]:
import plotly.express as px

fig = px.scatter_matrix(dfTemp,
    dimensions=dfTemp.columns,
                        symbol=dfTemp.index,
    title="Scatter matrix") # remove underscore
fig.update_traces(diagonal_visible=False)

fig.update_layout(
    xaxis=dict(type='log'),
    yaxis=dict(type='log'),)


In [ ]:
dfTemp.head()

In [ ]:
# x and y given as array_like objects
import plotly.express as px
fig = px.scatter(x=dfTemp['CShareD'], y=dfTemp['DShareC'])
fig.update_layout(
    xaxis=dict(type='log'),
    yaxis=dict(type='log'),)
fig.show()

# Create and html report

In [ ]:
text = '''<h1>Manuscript figures and tables - Global Context</h1>

Pearson correlations in the year 2024 with countries as entities
'''

In [ ]:
import seaborn as sns

def style_negative(v, props=''):
    return props if v < 0 else None

df = dictCorrAreasDF[2024].sort_values('F-Measure vs DShareC', ascending=False)
pd.set_option("styler.format.precision", 4)
cm = sns.light_palette("lightblue", as_cmap=True)

if(len(df) > 50):
  s = df.head(50).style.background_gradient(cmap=cm, vmin=0, vmax=1)
else:
  s = df.style.background_gradient(cmap=cm, vmin=0, vmax=1)
s.applymap(style_negative, props='color:red;')
text = text + s.to_html()

In [ ]:
import seaborn as sns

df = dictCorrAreasDFAGMeans[2024]
pd.set_option("styler.format.precision", 4)
cm = sns.light_palette("lightblue", as_cmap=True)

if(len(df) > 50):
  s = df.head(50).style.background_gradient(cmap=cm, vmin=0, vmax=1)
else:
  s = df.style.background_gradient(cmap=cm, vmin=0, vmax=1)
s.applymap(style_negative, props='color:red;')
text = text + s.to_html()

In [ ]:
text = text + '''
<h1>Spearman correlations in the year 2024 with countries as entities</h1>
'''

In [ ]:
import seaborn as sns

df = dictSpearmanCorrAreasDF[2024].sort_values('F-Measure vs DShareC', ascending=False)
pd.set_option("styler.format.precision", 4)
cm = sns.light_palette("lightblue", as_cmap=True)

#if(len(df) > 100):
#  s = df.head(100).style.background_gradient(cmap=cm)
#else:
#  s = df.style.background_gradient(cmap=cm)

s = df.style.background_gradient(cmap=cm, vmin=0, vmax=1)
s.applymap(style_negative, props='color:red;')
text = text + s.to_html()

In [ ]:
import seaborn as sns

df = dictSpearmanCorrAreasDFAGMeans[2024]
pd.set_option("styler.format.precision", 4)
cm = sns.light_palette("lightblue", as_cmap=True)

#if(len(df) > 100):
#  s = df.head(100).style.background_gradient(cmap=cm)
#else:
#  s = df.style.background_gradient(cmap=cm)

s = df.style.background_gradient(cmap=cm, vmin=0, vmax=1)
s.applymap(style_negative, props='color:red;')
text = text + s.to_html()

### Charts

In [ ]:
dictYearlyData[year][area]

In [ ]:
import plotly.express as px
import plotly.graph_objs as go

# Tnaks to user171780 in https://stackoverflow.com/questions/61494278/plotly-how-to-make-a-figure-with-multiple-lines-and-shaded-area-for-standard-de
def line(error_y_mode=None, **kwargs):
    """Extension of `plotly.express.line` to use error bands."""
    ERROR_MODES = {'bar','band','bars','bands',None}
    if error_y_mode not in ERROR_MODES:
        raise ValueError(f"'error_y_mode' must be one of {ERROR_MODES}, received {repr(error_y_mode)}.")
    if error_y_mode in {'bar','bars',None}:
        fig = px.line(**kwargs)
    elif error_y_mode in {'band','bands'}:
        if 'error_y' not in kwargs:
            raise ValueError(f"If you provide argument 'error_y_mode' you must also provide 'error_y'.")
        figure_with_error_bars = px.line(**kwargs)
        fig = px.line(**{arg: val for arg,val in kwargs.items() if arg != 'error_y'})
        for data in figure_with_error_bars.data:
            x = list(data['x'])
            y_upper = list(data['y'] + data['error_y']['array'])
            y_lower = list(data['y'] - data['error_y']['array'] if data['error_y']['arrayminus'] is None else data['y'] - data['error_y']['arrayminus'])
            color = f"rgba({tuple(int(data['line']['color'].lstrip('#')[i:i+2], 16) for i in (0, 2, 4))},.3)".replace('((','(').replace('),',',').replace(' ','')
            fig.add_trace(
                go.Scatter(
                    x = x+x[::-1],
                    y = y_upper+y_lower[::-1],
                    fill = 'toself',
                    fillcolor = color,
                    line = dict(
                        color = 'rgba(255,255,255,0)'
                    ),
                    hoverinfo = "skip",
                    showlegend = False,
                    legendgroup = data['legendgroup'],
                    xaxis = data['xaxis'],
                    yaxis = data['yaxis'],
                )
            )
        # Reorder data as said here: https://stackoverflow.com/a/66854398/8849755
        reordered_data = []
        fig.update_layout(hovermode="x")
        for i in range(int(len(fig.data)/2)):
            reordered_data.append(fig.data[i+int(len(fig.data)/2)])
            reordered_data.append(fig.data[i])
        fig.data = tuple(reordered_data)
    return fig

# This function receives a dataframe containing de correlations
def makeMeanStdSeries(df, cols):
  dfResult = pd.DataFrame(columns=['Name', 'Year', 'Mean', 'Std'])
  for year in years:
    for col in cols:
      dft = df[year][[col]]
      dftDesception = dft.describe()
      dfResult.loc[len(dfResult.index)] = [col, year, dftDesception.at['mean', col], dftDesception.at['std', col]]
  return dfResult

def makeSeries(df, col):
  dfResult = pd.DataFrame(columns=['Domain Area', 'Year', col])
  for year in years:
    dft = df[year][[col]]
    for area in areas:
      dfResult.loc[len(dfResult.index)] = [area, year, dft.at[area, col]]#Agregar un renglon
  return dfResult

def makeAreaSeries(df, col):

  dfT = df[2024][[col]]
  minV = dfT.min().values[0]
  areaMin = dfT[dfT[col] == minV].index[0]
  maxV = dfT.max().values[0]
  areaMax = dfT[dfT[col] == maxV].index[0]

  dfResult = pd.DataFrame(columns=['Domain', 'Year', col])
  for year in years:
    dfAreaMax = df[year][[col]].loc[[areaMax]]
    dfAreaMax['Year'] = year
    dfAreaMax['Domain'] = areaMax
    dfResult = pd.concat([dfResult, dfAreaMax])
    dfAreaMin = df[year][[col]].loc[[areaMin]]
    dfAreaMin['Year'] = year
    dfAreaMin['Domain'] = areaMin
    dfResult = pd.concat([dfResult, dfAreaMin])
  return dfResult


In [ ]:
df = makeSeries(dictCorrAreasDF, 'F-Measure vs DShareC')
df = df[df['Year'].isin([1980, 1990, 2000, 2010, 2020])]
fig = px.box(df, y="F-Measure vs DShareC", x="Year", color="Year",  points="all", title = 'Pearson Correlation',
          hover_data=df.columns)
fig.update_layout(
    font=dict(
        size=18,
    )
)
fig.show()

figName = 'figuras/pearson_correlation_longitudinal_fmeasure_nshare'


figNamehtml = figName+'.html'

#html
fig.write_html(figNamehtml,
              full_html=True,
              include_plotlyjs='cdn')
text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''

In [ ]:
df = makeSeries(dictSpearmanCorrAreasDF, 'F-Measure vs DShareC')
df = df[df['Year'].isin([1980, 1990, 2000, 2010, 2020])]
fig = px.box(df, y="F-Measure vs DShareC", x="Year", color="Year",  points="all", title = 'Spearman Correlation',
          hover_data=df.columns)
fig.update_layout(
    font=dict(
        size=18,
    ))
fig.show()

figName = 'figuras/spearman_correlation_longitudinal_fmeasure_nshare'
figNamehtml = figName+'.html'

#html
fig.write_html(figNamehtml,
              full_html=False,
              include_plotlyjs='cdn')
text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''

In [ ]:
df = makeSeries(dictCorrAreasDF, 'F-Measure vs CShareD')
df = df[df['Year'].isin([1980, 1990, 2000, 2010, 2020])]
fig = px.box(df, y="F-Measure vs CShareD", x="Year", color="Year",  points="all", title = 'Pearson Correlation',
          hover_data=df.columns)
fig.update_layout(
    font=dict(
        size=18,
    ))
fig.show()

figName = 'figuras/pearson_correlation_longitudinal_fmeasure_wshare'


figNamehtml = figName+'.html'

#html
fig.write_html(figNamehtml,
              full_html=False,
              include_plotlyjs='cdn')
text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''

In [ ]:
df =makeSeries(dictSpearmanCorrAreasDF, 'F-Measure vs DShareC')
df=df[df['Year']==2024]
df = df[['Domain Area', 'F-Measure vs DShareC']]
df = df.sort_values(by='F-Measure vs DShareC', ascending=False)
df

In [ ]:
df.describe()

In [ ]:
df = makeSeries(dictSpearmanCorrAreasDF, 'F-Measure vs CShareD')
df = df[df['Year'].isin([1980, 1990, 2000, 2010, 2020])]
fig = px.box(df, y="F-Measure vs CShareD", x="Year", color="Year",  points="all", title = 'Spearman Correlation',
          hover_data=df.columns)
fig.update_layout(
    font=dict(
        size=18,
    ))
fig.show()

figName = 'figuras/spearman_correlation_longitudinal_fmeasure_wshare'


figNamehtml = figName+'.html'

#html
fig.write_html(figNamehtml,
              full_html=False,
              include_plotlyjs='cdn')

text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''

In [ ]:
dfT = makeMeanStdSeries(dictCorrAreasDF, ['F-Measure vs DShareC', 'F-Measure vs CShareD'])

fig = line(
        data_frame = dfT,
        x = 'Year',
        y = 'Mean',
        error_y = 'Std',
        error_y_mode = 'bands', # Here you say `band` or `bar`.
        color = 'Name',
        title = 'Pearson Correlation',
        markers = '.',
    )
fig.show()

figName = 'figuras/pearson_correlation_evolution_fmeasure'

figNamehtml = figName+'.html'

#html
fig.write_html(figNamehtml,
              full_html=False,
              include_plotlyjs='cdn')
text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''

In [ ]:
dfT = makeMeanStdSeries(dictSpearmanCorrAreasDF, ['F-Measure vs DShareC', 'F-Measure vs CShareD'])

fig = line(
        data_frame = dfT,
        x = 'Year',
        y = 'Mean',
        error_y = 'Std',
        error_y_mode = 'bands', # Here you say `band` or `bar`.
        color = 'Name',
        title = 'Spearman Correlation',
        markers = '.',
    )
fig.show()

figName = 'figuras/spearman_correlation_evolution_fmeasure'


figNamehtml = figName+'.html'
#html
fig.write_html(figNamehtml,
              full_html=False,
              include_plotlyjs='cdn')
text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''

In [ ]:
dfT = makeMeanStdSeries(dictCorrAreasDF, ['Geometric Aggregtion vs DShareC', 'Geometric Aggregtion vs CShareD'])

fig = line(
        data_frame = dfT,
        x = 'Year',
        y = 'Mean',
        error_y = 'Std',
        error_y_mode = 'bands', # Here you say `band` or `bar`.
        color = 'Name',
        title = 'Pearson Correlation',
        markers = '.',
    )
fig.show()

figName = 'figuras/pearson_correlation_evolution_Geometric_aggregtion'


figNamehtml = figName+'.html'
#html
fig.write_html(figNamehtml,
              full_html=False,
              include_plotlyjs='cdn')
text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''

In [ ]:
dfT = makeMeanStdSeries(dictCorrAreasDF, ['AI vs DShareC', 'AI vs CShareD'])

fig = line(
        data_frame = dfT,
        x = 'Year',
        y = 'Mean',
        error_y = 'Std',
        error_y_mode = 'band', # Here you say `band` or `bar`.
        color = 'Name',
        title = 'Pearson Correlation',
        markers = '.',
    )
fig.show()

figName = 'figuras/pearson_correlation_evolution_AI'


figNamehtml = figName+'.html'

#html
fig.write_html(figNamehtml,
              full_html=False,
              include_plotlyjs='cdn')
text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''

In [ ]:
dfT = makeMeanStdSeries(dictSpearmanCorrAreasDF, ['AI vs DShareC', 'AI vs CShareD'])

fig = line(
        data_frame = dfT,
        x = 'Year',
        y = 'Mean',
        error_y = 'Std',
        error_y_mode = 'band', # Here you say `band` or `bar`.
        color = 'Name',
        title = 'Spearman Correlation',
        markers = '.',
    )
fig.show()

figName = 'figuras/spearman_correlation_evolution_AI'


figNamehtml = figName+'.html'

#html
fig.write_html(figNamehtml,
              full_html=False,
              include_plotlyjs='cdn')
text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''

In [ ]:
df = makeSeries(dictCorrAreasDF, 'AI vs CShareD')
df = df[df['Year'].isin([1980, 1990, 2000, 2010, 2020])]
fig = px.box(df, y="AI vs CShareD", x="Year", color="Year",  points="all", title = 'Pearson Correlation',
          hover_data=df.columns)
fig.show()

figName = 'figuras/pearson_correlation_longitudinal_AI_wshare'

figNamehtml = figName+'.html'

#html
fig.write_html(figNamehtml,
              full_html=False,
              include_plotlyjs='cdn')
text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''

In [ ]:

dfT = makeMeanStdSeries(dictCorrAreasDFAGMeans, ['AMean vs DShareC', 'AMean vs CShareD'])

fig = line(
        data_frame = dfT,
        x = 'Year',
        y = 'Mean',
        error_y = 'Std',
        error_y_mode = 'bands', # Here you say `band` or `bar`.
        color = 'Name',
        title = 'Pearson Correlation',
        markers = '.',
    )
fig.show()

figName = 'figuras/pearson_correlation_evolution_A-Measure'


figNamehtml = figName+'.html'

#html
fig.write_html(figNamehtml,
              full_html=False,
              include_plotlyjs='cdn')
text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''

In [ ]:

dfT = makeMeanStdSeries(dictSpearmanCorrAreasDFAGMeans, ['AMean vs DShareC', 'AMean vs CShareD'])

fig = line(
        data_frame = dfT,
        x = 'Year',
        y = 'Mean',
        error_y = 'Std',
        error_y_mode = 'bands', # Here you say `band` or `bar`.
        color = 'Name',
        title = 'Spearman Correlation',
        markers = '.',
    )
fig.show()


figName = 'figuras/spearman_correlation_evolution_A-Measure'


figNamehtml = figName+'.html'

#html
fig.write_html(figNamehtml,
              full_html=False,
              include_plotlyjs='cdn')
text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''

In [ ]:

dfT = makeMeanStdSeries(dictCorrAreasDFAGMeans, ['GMean vs DShareC', 'GMean vs CShareD'])

fig = line(
        data_frame = dfT,
        x = 'Year',
        y = 'Mean',
        error_y = 'Std',
        error_y_mode = 'bands', # Here you say `band` or `bar`.
        color = 'Name',
        title = 'Pearson Correlation',
        markers = '.',
    )
fig.show()


figName = 'figuras/pearson_correlation_evolution_G-Measure'


figNamehtml = figName+'.html'

#html
fig.write_html(figNamehtml,
              full_html=False,
              include_plotlyjs='cdn')
text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''

In [ ]:
dfT = makeMeanStdSeries(dictSpearmanCorrAreasDFAGMeans, ['GMean vs DShareC', 'GMean vs CShareD'])

fig = line(
        data_frame = dfT,
        x = 'Year',
        y = 'Mean',
        error_y = 'Std',
        error_y_mode = 'bands', # Here you say `band` or `bar`.
        color = 'Name',
        title = 'Spearman Correlation',
        markers = '.',
    )
fig.show()


figName = 'figuras/spearman_correlation_evolution_G-Measure'

figNamehtml = figName+'.html'

#html
fig.write_html(figNamehtml,
              full_html=False,
              include_plotlyjs='cdn')
text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''

In [ ]:
import plotly.express as px

col = 'F-Measure vs DShareC'
df = makeAreaSeries(dictCorrAreasDF, col)

fig = px.line(df, x="Year", y=col, color='Domain', title = 'Pearson Correlation', markers = '.')
fig.show()


figName = 'figuras/pearson_correlation_evolution_FMeasureDShareC2Areas'

figNamehtml = figName+'.html'

#html
fig.write_html(figNamehtml,
              full_html=False,
              include_plotlyjs='cdn')
text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''


In [ ]:
import plotly.express as px

col = 'F-Measure vs DShareC'
df = makeAreaSeries(dictSpearmanCorrAreasDF, col)

fig = px.line(df, x="Year", y=col, color='Domain', title = 'Spearman Correlation', markers = '.')
fig.show()

figName = 'figuras/spearman_correlation_best_worst_fmeasure_nshare'

figNamehtml = figName+'.html'
#html
fig.write_html(figNamehtml,
              full_html=False,
              include_plotlyjs='cdn')
text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''

In [ ]:
col = 'AI vs CShareD'
df = makeAreaSeries(dictSpearmanCorrAreasDF, col)

fig = px.line(df, x="Year", y=col, color='Domain', title = 'Spearman Correlation', markers = '.')
fig.show()

figName = 'figuras/spearman_correlation_best_worst_ai_wshare'


figNamehtml = figName+'.html'

#html
fig.write_html(figNamehtml,
              full_html=False,
              include_plotlyjs='cdn')
text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''

In [ ]:
import plotly.express as px

df = makeSeries(dictCorrAreasDF, 'F-Measure vs DShareC')
fig = px.line(df, x="Year", y="F-Measure vs DShareC", color='Domain Area',  title = 'Pearson Correlation', markers = '.')
fig.show()

figName = 'figuras/pearson_correlation_fmeasure_DShareC_alldomains'


figNamehtml = figName+'.html'

#html
fig.write_html(figNamehtml,
              full_html=False,
              include_plotlyjs='cdn')
text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''

In [ ]:
import plotly.express as px

df = makeSeries(dictCorrAreasDF, 'F-Measure vs CShareD')
fig = px.line(df, x="Year", y="F-Measure vs CShareD", color='Domain Area', title = 'Pearson Correlation', markers = '.')
fig.show()

figName = 'figuras/pearson_correlation_fmeasure_CShareD_alldomains'

figNamehtml = figName+'.html'

#html
fig.write_html(figNamehtml,
              full_html=False,
              include_plotlyjs='cdn')
text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''

In [ ]:
import plotly.express as px

df = makeSeries(dictCorrAreasDF, 'AI vs DShareC')
fig = px.line(df, x="Year", y="AI vs DShareC", color='Domain Area',  title = 'Pearson Correlation', markers = '.')
fig.show()


figName = 'figuras/pearson_correlation_AI_DShareC_alldomains'

figNamehtml = figName+'.html'

#html
fig.write_html(figNamehtml,
              full_html=False,
              include_plotlyjs='cdn')
text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''

In [ ]:
import plotly.express as px

df = makeSeries(dictCorrAreasDF, 'AI vs CShareD')
fig = px.line(df, x="Year", y="AI vs CShareD", color='Domain Area',  title = 'Pearson Correlation', markers = '.')
fig.show()

figName = 'figuras/pearson_correlation_AI_CShareD_alldomains'

figNamehtml = figName+'.html'

#html
fig.write_html(figNamehtml,
              full_html=False,
              include_plotlyjs='cdn')
text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''

In [ ]:
import plotly.express as px

df = makeSeries(dictSpearmanCorrAreasDF, 'F-Measure vs DShareC')
fig = px.line(df, x="Year", y="F-Measure vs DShareC", color='Domain Area', title = 'Spearman Correlation', markers = '.')
fig.show()

figName = 'figuras/Spearman_correlation_fmeasure_DShareC_alldomains'

figNamehtml = figName+'.html'

#html
fig.write_html(figNamehtml,
              full_html=False,
              include_plotlyjs='cdn')
text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''

In [ ]:
import plotly.express as px

df = makeSeries(dictSpearmanCorrAreasDF, 'F-Measure vs CShareD')
fig = px.line(df, x="Year", y="F-Measure vs CShareD", color='Domain Area', title = 'Spearman Correlation', markers = '.')
fig.show()

figName = 'figuras/Spearman_correlation_fmeasure_CShareD_alldomains'

figNamehtml = figName+'.html'

#html
fig.write_html(figNamehtml,
              full_html=False,
              include_plotlyjs='cdn')
text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''

In [ ]:
import plotly.express as px

df = makeSeries(dictSpearmanCorrAreasDF, 'AI vs DShareC')
fig = px.line(df, x="Year", y="AI vs DShareC", color='Domain Area', title = 'Spearman Correlation', markers = '.')
fig.show()

figName = 'figuras/Spearman_correlation_AI_DShareC_alldomains'

figNamehtml = figName+'.html'

#html
fig.write_html(figNamehtml,
              full_html=False,
              include_plotlyjs='cdn')
text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''

In [ ]:
import plotly.express as px

df = makeSeries(dictSpearmanCorrAreasDF, 'AI vs CShareD')
fig = px.line(df, x="Year", y="AI vs CShareD", color='Domain Area', title = 'Spearman Correlation', markers = '.')
fig.show()

figName = 'figuras/Spearman_correlation_AI_CShareD_alldomains'

figNamehtml = figName+'.html'

#html
fig.write_html(figNamehtml,
              full_html=False,
              include_plotlyjs='cdn')
text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''

# Tables

In [ ]:
print(year)
dictCorrAreas[year][area].style.format(precision=4)

In [ ]:
dictCorrAreasDF[2024].style.format(precision=4)

In [ ]:
dictCorrAreasDF[2022].style.format(precision=4)

In [ ]:
dictCorrAreasDF[year].describe().style.format(precision=4)

In [ ]:
dictCorrAreasDF[1980].style.format(precision=4)

In [ ]:
dictSpearmanCorrAreasDF[year].style.format(precision=3)

In [ ]:
dictSpearmanCorrAreasDF[year].describe().style.format(precision=4)

In [ ]:
dictCorrAreasDFAGMeans[year].style.format(precision=4)

In [ ]:
dictCorrAreasDFAGMeans[year].describe().style.format(precision=4)

In [ ]:
dictSpearmanCorrAreasDFAGMeans[year].style.format(precision=4)

In [ ]:
dictSpearmanCorrAreasDFAGMeans[year].describe().style.format(precision=4)

In [ ]:
dictCorrAreas[2024]['Clinical Medicine']

In [ ]:
dictCorrAreas[1980]['Clinical Medicine']

In [ ]:
dictCorrAreas[2024]['Mathematics']

In [ ]:
dictCorrAreas[1980]['Mathematics']

In [ ]:
dictCorrAreas[2024]['Space Science']

In [ ]:
dictCorrAreas[1980]['Space Science']

In [ ]:
text = text + '''</div>'''
file = open("manuscriptFiguresAndTables.html","w")
file.write(text)
file.close()